In [ ]:
"""
Download the public IBC dataset (Zenodo DOI: 10.5281/zenodo.8214497)
and unzip it into data/raw/.

Usage
-----
python data/download_dataset.py
"""
import pathlib, requests, zipfile, io

RECORD_ID = 8214497
ZIP_URL   = f"https://zenodo.org/api/records/{RECORD_ID}/files-archive"
DEST_DIR  = pathlib.Path("data/raw")
DEST_DIR.mkdir(parents=True, exist_ok=True)

print("Downloading dataset ZIP…")
resp = requests.get(ZIP_URL, timeout=60)
resp.raise_for_status()

print("Extracting…")
with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
    zf.extractall(DEST_DIR)

print("✔ Done. CSV is in data/raw/all_measurements.csv")

In [ ]:
"""
data/preprocess.py
Pre-processing pipeline for the IBC benchmark.

Steps
1) load all_measurements.csv
2) drop NaNs
3) remove >3 σ outliers (per-frequency column)
4) 400-point → 256-point linear interpolation
5) per-spectrum z-score normalisation
6) save to data/processed/ibc_processed.csv
"""
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d

# ────────────────────────────────────────────
RAW_CSV   = Path("data/raw/all_measurements.csv")   # ← 첨부 CSV 위치
OUT_DIR   = Path("data/processed")
OUT_CSV   = OUT_DIR / "ibc_processed.csv"
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_RAW, N_NEW = 400, 256
F_RAW   = np.linspace(50e3, 20e6, N_RAW)        # 50 kHz – 20 MHz
F_NEW   = np.linspace(50e3, 20e6, N_NEW)
# ────────────────────────────────────────────


def clean_outliers(df: pd.DataFrame) -> pd.DataFrame:
    """3 σ trimming, column-wise."""
    numeric = df.iloc[:, 1:]                     # 첫 컬럼은 메타데이터(예: subject_id)
    mask_lo = (numeric - numeric.mean()) >= -3 * numeric.std()
    mask_hi = (numeric - numeric.mean()) <=  3 * numeric.std()
    return df[mask_lo & mask_hi].dropna()


def interpolate_row(row: np.ndarray) -> np.ndarray:
    interp_fn = interp1d(F_RAW, row, kind="linear")
    return interp_fn(F_NEW)


def main() -> None:
    df = pd.read_csv(RAW_CSV)
    print(f"▶ loaded {len(df):,} spectra")

    df = df.dropna()
    df = clean_outliers(df)
    print(f"▶ {len(df):,} spectra after cleaning")

    spectra = np.vstack(df.iloc[:, 1:].apply(interpolate_row, axis=1).to_numpy())
    spectra = (spectra - spectra.mean(1, keepdims=True)) / spectra.std(1, keepdims=True)

    pd.DataFrame(spectra, columns=[f"f_{i}" for i in range(N_NEW)]).to_csv(OUT_CSV, index=False)
    print(f"✔ saved pre-processed spectra → {OUT_CSV}")


if __name__ == "__main__":
    main()


In [ ]:
"""
data/preprocess.py
Pre-processing pipeline for the IBC benchmark.

Workflow
1.  Load raw spectra from data/raw/all_measurements.csv
2.  Drop NaNs
3.  Remove rows that contain >3 σ outliers (column-wise)
4.  Interpolate each spectrum from N_raw (71) → 256 points
5.  Per-spectrum z-score normalisation
6.  Save to data/processed/ibc_processed.csv
"""

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d

# ---------------------------------------------------------------------
RAW_CSV = Path("data/raw/all_measurements.csv")
OUT_CSV = Path("data/processed/ibc_processed.csv")
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

SIGMA = 3               # outlier threshold
N_TARGET = 256          # points after resampling
F_START, F_STOP = 50e3, 20e6   # 50 kHz – 20 MHz
# ---------------------------------------------------------------------


def trim_outliers(df: pd.DataFrame, sigma: float) -> pd.DataFrame:
    """Remove any row that contains a value outside ±sigma σ in *any* column."""
    num = df.iloc[:, 1:]                       # skip metadata column 0
    mask = (np.abs(num - num.mean()) <= sigma * num.std()).all(axis=1)
    return df[mask]


def interp_spectrum(y: np.ndarray, f_raw: np.ndarray, f_new: np.ndarray) -> np.ndarray:
    """Interpolate one spectrum onto the target frequency grid."""
    if y.size != f_raw.size:
        raise ValueError(f"Spectrum has {y.size} points but f_raw has {f_raw.size}")
    return interp1d(f_raw, y, kind="linear")(f_new)


def main() -> None:
    if not RAW_CSV.exists():
        raise FileNotFoundError(RAW_CSV)

    df = pd.read_csv(RAW_CSV)
    print(f"Loaded raw file           : {df.shape}")

    df = df.dropna()
    print(f"After dropna              : {df.shape}")

    df = trim_outliers(df, SIGMA)
    if df.empty:
        raise RuntimeError("All rows removed by outlier trimming")
    print(f"After {SIGMA}σ trim           : {df.shape}")

    # Determine actual spectral length dynamically (metadata col + spectra cols)
    n_raw = df.shape[1] - 1
    f_raw = np.linspace(F_START, F_STOP, n_raw)
    f_new = np.linspace(F_START, F_STOP, N_TARGET)

    # Interpolate and normalise
    spectra = np.vstack(
        df.iloc[:, 1:].apply(
            lambda row: interp_spectrum(row.to_numpy(), f_raw, f_new), axis=1
        )
    )
    spectra = (spectra - spectra.mean(1, keepdims=True)) / spectra.std(
        1, keepdims=True
    )

    pd.DataFrame(
        spectra, columns=[f"f_{i}" for i in range(N_TARGET)]
    ).to_csv(OUT_CSV, index=False)
    print(f"Saved processed spectra → {OUT_CSV}")


if __name__ == "__main__":
    main()


In [ ]:
"""
Simple-3 feature extraction for IBC spectra
d  : electrode distance (cm)
g50: gain @12.5 MHz, 50 Ω load
g1M: gain @12.5 MHz, 1 MΩ load
"""
from pathlib import Path
import pandas as pd
import numpy as np

INPUT_CSV  = Path("data/processed/ibc_processed.csv")
OUTPUT_CSV = Path("features/simple3_features.csv")
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------

def main() -> None:
    df = pd.read_csv(INPUT_CSV)

    # frequency axis of the processed spectra (256 points, 50 kHz–20 MHz)
    f_start, f_stop, n_points = 50e3, 20e6, df.shape[1]
    f_axis = np.linspace(f_start, f_stop, n_points)
    idx = np.argmin(np.abs(f_axis - 12.5e6))

    d = 10  # cm – replace with actual value if variable

    features = pd.DataFrame({
        "d":   np.full(len(df), d),
        "g50": df.iloc[:, idx],   # gain @12.5 MHz, 50 Ω
        "g1M": df.iloc[:, idx],   # gain @12.5 MHz, 1 MΩ (same col if not separated)
    })
    features.to_csv(OUTPUT_CSV, index=False)
    print(f"Saved Simple-3 features → {OUTPUT_CSV}")

if __name__ == "__main__":
    main()


In [ ]:
"""
Discrete Wavelet Transform (Daubechies-4, level 2) features:
for every sub-band (b0-b2) compute energy, entropy, mean, std.
"""
from pathlib import Path
import pandas as pd
import numpy as np
import pywt

INPUT_CSV  = Path("data/processed/ibc_processed.csv")
OUTPUT_CSV = Path("features/dwt_db4_features.csv")
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

LEVEL   = 2
WAVELET = "db4"
STATS   = ("energy", "entropy", "mean", "std")

def band_stats(coeff) -> list[float]:
    energy = float(np.sum(coeff ** 2))
    prob   = coeff**2 / energy if energy else np.zeros_like(coeff)
    entropy = float(-np.sum(prob * np.log2(prob + 1e-12)))
    return [energy, entropy, float(coeff.mean()), float(coeff.std())]

def main() -> None:
    df = pd.read_csv(INPUT_CSV)
    rows = []
    for _, row in df.iterrows():
        coeffs = pywt.wavedec(row.values, WAVELET, level=LEVEL, mode="periodization")
        rows.append([x for c in coeffs for x in band_stats(c)])

    cols = [f"b{b}_{s}" for b in range(LEVEL + 1) for s in STATS]
    pd.DataFrame(rows, columns=cols).to_csv(OUTPUT_CSV, index=False)
    print(f"Saved DWT-db4 features → {OUTPUT_CSV}")

if __name__ == "__main__":
    main()


In [ ]:
"""
Lifting-scheme biorthogonal (bior2.2, level 2) features.
Same statistics as DWT script.
"""
from pathlib import Path
import pandas as pd
import numpy as np
import pywt

INPUT_CSV  = Path("data/processed/ibc_processed.csv")
OUTPUT_CSV = Path("features/lift_bior_features.csv")
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

LEVEL   = 2
WAVELET = "bior2.2"
STATS   = ("energy", "entropy", "mean", "std")

def band_stats(coeff) -> list[float]:
    energy  = float(np.sum(coeff ** 2))
    prob    = coeff**2 / energy if energy else np.zeros_like(coeff)
    entropy = float(-np.sum(prob * np.log2(prob + 1e-12)))
    return [energy, entropy, float(coeff.mean()), float(coeff.std())]

def main() -> None:
    df = pd.read_csv(INPUT_CSV)
    rows = []
    for _, row in df.iterrows():
        coeffs = pywt.wavedec(row.values, WAVELET, level=LEVEL)
        rows.append([x for c in coeffs for x in band_stats(c)])

    cols = [f"b{b}_{s}" for b in range(LEVEL + 1) for s in STATS]
    pd.DataFrame(rows, columns=cols).to_csv(OUTPUT_CSV, index=False)
    print(f"Saved lifting-bior features → {OUTPUT_CSV}")

if __name__ == "__main__":
    main()


In [ ]:
"""
Second-order 1-D scattering transform (Kymatio).

Parameters
----------
J : log2(max scale)
Q : number of wavelets per octave
"""
from pathlib import Path
import pandas as pd
import numpy as np
from kymatio.numpy import Scattering1D

INPUT_CSV  = Path("data/processed/ibc_processed.csv")
OUTPUT_CSV = Path("features/scattering_features.csv")
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

J, Q = 5, 8   # follows the paper

def main() -> None:
    df = pd.read_csv(INPUT_CSV)
    signal_len = df.shape[1]
    scat = Scattering1D(J=J, shape=signal_len, Q=Q)
    features = scat(df.values).reshape(df.shape[0], -1)
    col_names = [f"s_{i}" for i in range(features.shape[1])]
    pd.DataFrame(features, columns=col_names).to_csv(OUTPUT_CSV, index=False)
    print(f"Saved scattering features → {OUTPUT_CSV}")

if __name__ == "__main__":
    main()


In [ ]:
"""
Convenience launcher:
python -m features.run --method dwt
"""
import argparse, importlib

REGISTRY = {
    "simple3":  "features.simple3",
    "dwt":      "features.dwt_db4",
    "lift":     "features.lift_bior",
    "scatter":  "features.scattering",
}

def main() -> None:
    p = argparse.ArgumentParser()
    p.add_argument("--method", choices=REGISTRY, required=True)
    args = p.parse_args()

    module = importlib.import_module(REGISTRY[args.method])
    module.main()

if __name__ == "__main__":
    main()


In [ ]:
"""
Create labels_filtered.csv that is row-synchronised with the
pre-processed spectra (after NaN removal and 3σ trimming).
"""
from pathlib import Path
import pandas as pd
import numpy as np

RAW_CSV   = Path("data/raw/all_measurements.csv")
LABEL_CSV = Path("data/labels_filtered.csv")
LABEL_CSV.parent.mkdir(parents=True, exist_ok=True)

SIGMA = 3                   # keep identical to preprocess.py

def main() -> None:
    df = pd.read_csv(RAW_CSV)

    # Standardise the subject-ID column name
    if "subject_id" not in df.columns:
        df.rename(columns={df.columns[0]: "subject_id"}, inplace=True)

    # 1.  drop NaNs
    df = df.dropna()

    # 2.  3σ trimming (identical to preprocess.py logic)
    num = df.iloc[:, 1:]                     # spectra only
    mask = (np.abs(num - num.mean()) <= SIGMA * num.std()).all(axis=1)
    df = df[mask]

    # 3.  save filtered labels
    df[["subject_id"]].to_csv(LABEL_CSV, index=False)
    print(f"✔  filtered labels written → {LABEL_CSV}")

if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
from collections import Counter

X = pd.read_csv("features/dwt_db4_features.csv")
y = pd.read_csv("data/labels_filtered.csv")["subject_id"]

print("Shapes →", X.shape, y.shape)        # 둘 다 (941, …) 여야 함
print("Unique labels →", len(set(y)))      # 예상 30개
print("Per-class counts →", Counter(y))


In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import KNeighborsClassifier

gss = GroupShuffleSplit(test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=y))

knn = KNeighborsClassifier(n_neighbors=1)
knn.fit(X.iloc[train_idx], y.iloc[train_idx])
print("KNN accuracy =", knn.score(X.iloc[test_idx], y.iloc[test_idx]))


In [ ]:
import pandas as pd
from collections import Counter

X = pd.read_csv("features/dwt_db4_features.csv")
y = pd.read_csv("data/labels_filtered.csv")["subject_id"]

print("Shapes →", X.shape, y.shape)        # 둘 다 (941, …) 여야 함
print("Unique labels →", len(set(y)))      # 예상 30개
print("Per-class counts →", Counter(y))


In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import KNeighborsClassifier

gss = GroupShuffleSplit(test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=y))

knn = KNeighborsClassifier(n_neighbors=1)
knn.fit(X.iloc[train_idx], y.iloc[train_idx])
print("KNN accuracy =", knn.score(X.iloc[test_idx], y.iloc[test_idx]))


In [ ]:
from collections import Counter
print("label counts:", Counter(y))

print("feature means (first 5):", X.mean(axis=0)[:5])
print("feature stds  (first 5):", X.std(axis=0)[:5])


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedShuffleSplit

sss = StratifiedShuffleSplit(test_size=0.2, random_state=0)
tr, te = next(sss.split(X, y))
print("unique classes in test:", set(y[te]))

knn = KNeighborsClassifier(n_neighbors=1).fit(X.iloc[tr], y.iloc[tr])
print("1-NN accuracy:", knn.score(X.iloc[te], y.iloc[te]))


In [ ]:
from collections import Counter
print("Train:", Counter(y_train))
print("Test :", Counter(y_test))
